# Auto-validated binary pair specialist — seed42

## 가설
H0가 실제로 혼동한 암종쌍 중, inner OOF에서 binary specialist가 pair F1을 개선한 쌍만 보수적으로 적용하면 H0의 일부 경계 오류를 고칠 수 있습니다.

## 규칙 및 leakage 구조
- Outer: Stratified 5-fold seed42 / Inner: outer-train 내부 3-fold
- 암종쌍은 inner H0 OOF의 양방향 혼동에서 자동 탐지합니다. 고정 암종·유전자·mutation 목록은 없습니다.
- inner validation에서 pair F1 상승 및 `recovered > broken`인 쌍만, 서로 겹치지 않게 최대 2개 채택합니다.
- outer validation에는 선택·학습된 변환만 적용합니다. H0 Top-2가 선택 쌍과 정확히 일치할 때만 내부 확률 비율을 specialist로 바꾸며 pair 확률 질량은 보존합니다.
- test는 읽지 않고, train/test concat·test 통계·test encoding·threshold 탐색을 하지 않습니다. WT/blank/NaN은 event가 아닙니다.

전체 실행은 inner H0 OOF와 후보별 binary expert를 생성하므로 시간이 오래 걸릴 수 있습니다.

In [ ]:
from pathlib import Path
import subprocess, sys
from tqdm.auto import tqdm

ROOT = Path('/Users/admin/Documents/FinalProject/OZ_fianl_hackaton')
RUNNER = ROOT / 'experiments/gs/notebooks/exp_model_006/common/run_auto_validated_pair_specialist.py'
RESULT = ROOT / 'experiments/gs/notebooks/exp_model_006/result'
RUN_ID = 'exp-auto-validated-pair-specialist-01'
RUN_EXPERIMENT = True
assert RUNNER.exists()
print({'runner': RUNNER, 'result_dir': RESULT, 'outer_seed': 42, 'inner_splits': 3, 'max_selected_pairs': 2, 'test_read': False})

In [ ]:
if RUN_EXPERIMENT:
    process = subprocess.Popen([sys.executable, str(RUNNER), '--run-id', RUN_ID], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in tqdm(process.stdout, desc='auto pair specialist', unit='line'):
        print(line, end='')
        tail = (tail + [line])[-120:]
    if process.wait():
        raise RuntimeError('Auto-pair specialist runner failed:\n' + ''.join(tail))
else:
    print('RUN_EXPERIMENT=False: 기존 result만 읽습니다.')

In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd

summary = pd.read_csv(RESULT / f'{RUN_ID}_seed42_summary.csv')
folds = pd.read_csv(RESULT / f'{RUN_ID}_seed42_fold_metrics.csv')
candidates = pd.read_csv(RESULT / f'{RUN_ID}_seed42_inner_candidate_audit.csv')
applied = pd.read_csv(RESULT / f'{RUN_ID}_seed42_applied_pairs.csv')
classes = pd.read_csv(RESULT / f'{RUN_ID}_seed42_class_metrics.csv')
low_margin = pd.read_csv(RESULT / f'{RUN_ID}_seed42_low_margin.csv')
audit = json.loads((RESULT / f'{RUN_ID}_seed42_leakage_audit.json').read_text())
assert summary.leakage_check.all() and summary.nan_as_mutation_count.eq(0).all()
display(summary.sort_values('oof_macro_f1', ascending=False))
display(candidates.sort_values(['outer_fold', 'selected', 'pair_f1_delta'], ascending=[True, False, False]).head(30))
display(applied)
display(low_margin)
print('자동 판정:', audit['decision'])
print({key: audit[key] for key in ('delta_vs_h0', 'positive_fold_count', 'low_margin_delta', 'runtime_seconds')})

folds.pivot(index='fold', columns='variant', values='macro_f1').plot(marker='o', figsize=(8, 4), title='H0 vs auto-validated pair specialist')
plt.ylabel('Macro F1'); plt.tight_layout(); plt.show()
classes.set_index('class').delta.sort_values().plot.barh(figsize=(7, 7), title='Class F1 delta vs H0')
plt.tight_layout(); plt.show()